# 13 — Basket Options

Multi-asset options on a weighted basket:
- **Stulz** (2-asset, exact)
- **Moment-matching** (N-asset, approximate)
- **Single-factor BSM** basket
- **Kirk's spread** approximation
- **Monte Carlo** (N-asset)
- AD: Jacobian of basket price w.r.t. all spots

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms, plot_heatmap
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.engines.analytic.basket import (
    stulz_basket_price, moment_matching_basket_price, mc_european_basket)
from ql_jax.engines.analytic.basket_single_factor import (
    operator_splitting_spread_price, single_factor_bsm_basket_price)

# 2-asset parameters
S1, S2 = 100.0, 100.0
K = 100.0
T, r = 1.0, 0.05
q1, q2 = 0.02, 0.03
sigma1, sigma2 = 0.25, 0.30
rho = 0.5

---
## 1. Stulz Two-Asset Basket

In [ ]:
stulz_call = float(stulz_basket_price(S1, S2, K, T, r, q1, q2, sigma1, sigma2, rho, option_type=1))
stulz_put  = float(stulz_basket_price(S1, S2, K, T, r, q1, q2, sigma1, sigma2, rho, option_type=-1))

print(f"Stulz call on max(S1,S2) : {stulz_call:.6f}")
print(f"Stulz put on min(S1,S2)  : {stulz_put:.6f}")

---
## 2. N-Asset Moment-Matching

In [ ]:
# 4-asset basket
spots_4  = jnp.array([100.0, 105.0, 98.0, 102.0])
divs_4   = jnp.array([0.02, 0.01, 0.03, 0.015])
sigmas_4 = jnp.array([0.25, 0.30, 0.20, 0.28])
corr_4   = jnp.array([
    [1.0,  0.5,  0.3,  0.4],
    [0.5,  1.0,  0.6,  0.35],
    [0.3,  0.6,  1.0,  0.45],
    [0.4,  0.35, 0.45, 1.0]
])
weights_4 = jnp.array([0.25, 0.25, 0.25, 0.25])  # equal weight

mm_call = float(moment_matching_basket_price(
    spots_4, K, T, r, divs_4, sigmas_4, corr_4, weights_4, option_type=1))
mm_put = float(moment_matching_basket_price(
    spots_4, K, T, r, divs_4, sigmas_4, corr_4, weights_4, option_type=-1))

print(f"4-asset basket call (moment-matching) : {mm_call:.6f}")
print(f"4-asset basket put                    : {mm_put:.6f}")

---
## 3. Monte Carlo Basket

In [ ]:
mc_call = float(mc_european_basket(
    spots_4, K, T, r, divs_4, sigmas_4, corr_4,
    weights_4, option_type=1, n_paths=200_000, seed=42))

print(f"MC basket call (200k paths) : {mc_call:.6f}")
print(f"Moment-matching basket call : {mm_call:.6f}")
print(f"Difference                  : {abs(mc_call - mm_call):.6f}")

---
## 4. Spread Option (Kirk's Approximation)

In [ ]:
# Spread = F1 - F2 - K
F1, F2 = 100.0, 95.0  # forward prices
df = np.exp(-r * T)

spread_call = float(operator_splitting_spread_price(
    F1, F2, T, sigma1, sigma2, rho, K=5.0, df=df, option_type=1))
spread_put = float(operator_splitting_spread_price(
    F1, F2, T, sigma1, sigma2, rho, K=5.0, df=df, option_type=-1))

print(f"Spread call (F1-F2-K, K=5) : {spread_call:.6f}")
print(f"Spread put                 : {spread_put:.6f}")

---
## 5. Correlation Sensitivity

In [ ]:
import matplotlib.pyplot as plt

# 2-asset basket: price vs correlation
rhos = np.linspace(-0.9, 0.9, 30)
stulz_prices = [float(stulz_basket_price(S1, S2, K, T, r, q1, q2,
                       sigma1, sigma2, rho_i, option_type=1)) for rho_i in rhos]

plt.figure(figsize=(8, 5))
plt.plot(rhos, stulz_prices, 'bo-', markersize=4)
plt.xlabel('Correlation')
plt.ylabel('Call on max(S1,S2)')
plt.title('Stulz Basket Price vs Correlation')
plt.grid(True, alpha=0.3)
plt.show()

---
## 6. AD: Jacobian w.r.t. All Spots

In [ ]:
def basket_price_fn(spots):
    return moment_matching_basket_price(
        spots, K, T, r, divs_4, sigmas_4, corr_4, weights_4, option_type=1)

grad_fn = jax.grad(basket_price_fn)
deltas = grad_fn(spots_4)

print("Basket deltas (dV/dS_i):")
for i, d in enumerate(deltas):
    print(f"  Asset {i+1}: {float(d):.6f}")

# Hessian: cross-gammas
hessian = jax.hessian(basket_price_fn)(spots_4)

labels = ['S1', 'S2', 'S3', 'S4']
plot_heatmap(np.array(hessian), labels, labels)
plt.title('Basket Cross-Gamma (Hessian)')
plt.show()

---
## 7. Exercises

1. **Weighted basket**: Price with weights [0.5, 0.2, 0.2, 0.1] and compare to equal-weight.
2. **Best-of / Worst-of**: Use Stulz to price best-of-two vs worst-of-two calls.
3. **Correlation risk**: Compute $\partial V / \partial \rho$ for the spread option using AD.

---
*End of Notebook 13*